# Temperature Residuals Pipeline

Computes `residuals_df`: the difference between ECA weather station max temperatures and ERA5 reanalysis,
enriched with covariates (precipitation, wind, NDVI, distance to sea, DEGURBA urbanisation class).

Output: `../data2/residuals.parquet`

In [1]:
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
import rasterio
import xarray as xr
from rasterio.warp import transform
from tqdm import tqdm

In [2]:
DATA = Path("/home2/lobib/gen_hack/data/main")
OUT = Path("/home2/lobib/gen_hack/data2")
OUT.mkdir(exist_ok=True)

## 1. Load weather stations

In [3]:
def dms_to_decimal(dms_str):
    """Convert DMS string (+DD:MM:SS) to decimal degrees."""
    dms_str = dms_str.strip()
    sign = 1 if dms_str[0] == "+" else -1
    dms_str = dms_str[1:]
    parts = dms_str.split(":")
    return sign * (float(parts[0]) + float(parts[1]) / 60 + float(parts[2]) / 3600)

In [4]:
eca_tx_datafolder = DATA / "ECA_blend_tx/"
stations_filepath = eca_tx_datafolder / "stations.txt"

stations_df = pd.read_csv(stations_filepath, skiprows=17, skipinitialspace=True)
stations_df["LAT_decimal"] = stations_df["LAT"].apply(dms_to_decimal)
stations_df["LON_decimal"] = stations_df["LON"].apply(dms_to_decimal)

stations_gdf = gpd.GeoDataFrame(
    stations_df,
    geometry=gpd.points_from_xy(stations_df["LON_decimal"], stations_df["LAT_decimal"]),
    crs="EPSG:4326",
).drop(columns=["LAT", "LON", "LAT_decimal", "LON_decimal"])

print(f"Number of registered stations: {len(stations_gdf)}")
stations_gdf.head()

Number of registered stations: 8568


,STAID,STANAME,CN,HGHT,geometry
0,1,VAEXJOE,SE,166,POINT (14.8 56.86667)
1,2,FALUN,SE,160,POINT (15.61667 60.61667)
2,3,STENSELE,SE,325,POINT (17.16639 65.06667)
3,4,LINKOEPING,SE,93,POINT (15.53306 58.4)
4,5,LINKOEPING-MALMSLAETT,SE,93,POINT (15.53306 58.4)


## 2. Load ERA5 temperature and extract at station locations

In [5]:
era5_files = sorted(
    DATA.glob("derived-era5-land-daily-statistics/*_2m_temperature_daily_maximum.nc")
)
era5_ds = xr.open_mfdataset(era5_files, combine="by_coords")
era5_ds["t2m_celsius"] = era5_ds["t2m"] - 273.15

all_lats = xr.DataArray(stations_gdf.geometry.y.values, dims="station")
all_lons = xr.DataArray(stations_gdf.geometry.x.values, dims="station")

era5_all_stations = (
    era5_ds["t2m_celsius"]
    .sel(latitude=all_lats, longitude=all_lons, method="nearest")
    .compute()
)  # shape: (time, n_stations)

print(f"ERA5 shape: {era5_all_stations.shape}")

ERA5 shape: (2098, 8568)


## 3. Compute residuals (station - ERA5) per station

In [6]:
all_residuals = []
era5_times = pd.DatetimeIndex(era5_ds.valid_time.values)

for i, (_, row) in tqdm(
    enumerate(stations_gdf.iterrows()), total=len(stations_gdf), colour="green"
):
    filepath = eca_tx_datafolder / f"TX_STAID{row['STAID']:06d}.txt"
    if not filepath.exists():
        continue

    station_df = pd.read_csv(filepath, skiprows=20, skipinitialspace=True)
    station_df = station_df[station_df["Q_TX"] == 0].copy()
    station_df["DATE"] = pd.to_datetime(station_df["DATE"], format="%Y%m%d")
    station_df = station_df[station_df["DATE"].isin(era5_times)]

    if len(station_df) == 0:
        continue

    station_df["TX_celsius"] = station_df["TX"] / 10.0

    time_indices = era5_times.get_indexer(station_df["DATE"])
    era5_vals = era5_all_stations.values[time_indices, i]

    all_residuals.append(
        pd.DataFrame(
            {
                "STAID": row["STAID"],
                "date": station_df["DATE"].values,
                "station_celsius": station_df["TX_celsius"].values,
                "era5_celsius": era5_vals,
                "residual": station_df["TX_celsius"].values - era5_vals,
            }
        )
    )

residuals_df = pd.concat(all_residuals, ignore_index=True)
print(f"Residuals shape: {residuals_df.shape}")
residuals_df.head()

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 8568/8568 [02:21<00:00, 60.47it/s]


Residuals shape: (10467039, 5)


,STAID,date,station_celsius,era5_celsius,residual
0,2,2020-01-01,5.4,3.112396,2.287604
1,2,2020-01-02,6.3,4.385101,1.914899
2,2,2020-01-03,6.8,6.099335,0.700665
3,2,2020-01-04,2.7,1.308136,1.391864
4,2,2020-01-05,0.9,1.942871,-1.042871


## 4. Add ERA5 covariates (precipitation, wind)

In [7]:
variables = {
    "total_precipitation": ("daily_mean", "tp"),
    "10m_u_component_of_wind": ("daily_mean", "u10"),
    "10m_v_component_of_wind": ("daily_mean", "v10"),
}

staid_to_idx = {staid: i for i, staid in enumerate(stations_gdf["STAID"])}

for var_name, (statistic, datavar) in tqdm(
    variables.items(), total=len(variables), colour="green", position=0, leave=True
):
    print(f"Loading {datavar}...")
    files = sorted(
        DATA.glob(f"derived-era5-land-daily-statistics/*_{var_name}_{statistic}.nc")
    )
    ds = xr.open_mfdataset(files, combine="by_coords")

    extracted = (
        ds[datavar]
        .sel(latitude=all_lats, longitude=all_lons, method="nearest")
        .compute()
    )
    era5_var_times = pd.DatetimeIndex(ds.valid_time.values)

    time_indices = era5_var_times.get_indexer(residuals_df["date"])
    sta_indices = residuals_df["STAID"].map(staid_to_idx).values

    residuals_df[datavar] = extracted.values[time_indices, sta_indices]
    print(f"  {datavar}: {residuals_df[datavar].isna().sum()} nulls")
    ds.close()

# Compute wind speed (norm of u10, v10) and drop components
residuals_df["wind"] = np.sqrt(residuals_df["u10"] ** 2 + residuals_df["v10"] ** 2)
residuals_df.drop(columns=["u10", "v10"], inplace=True)
print(f"  wind nulls: {residuals_df['wind'].isna().sum()}")

  0%|                                                                                                                                                   | 0/3 [00:00<?, ?it/s]

Loading tp...


 33%|██████████████████████████████████████████████▎                                                                                            | 1/3 [00:45<01:30, 45.33s/it]

  tp: 1292521 nulls
Loading u10...


 67%|████████████████████████████████████████████████████████████████████████████████████████████▋                                              | 2/3 [01:28<00:44, 44.08s/it]

  u10: 1292521 nulls
Loading v10...


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 3/3 [02:11<00:00, 43.96s/it]

  v10: 1292521 nulls


  wind nulls: 1292521


## 5. Add NDVI (Sentinel-2, quarterly)

In [8]:
ndvi_dir = DATA / "sentinel2_ndvi/"


def quarter_tp(date):
    m, y = date.month, date.year
    if m < 3:
        return f"{y-1}-12-01_{y}-03-01"
    elif m < 6:
        return f"{y}-03-01_{y}-06-01"
    elif m < 9:
        return f"{y}-06-01_{y}-09-01"
    elif m < 12:
        return f"{y}-09-01_{y}-12-01"
    else:
        return f"{y}-12-01_{y+1}-03-01"


sta_lats = stations_gdf.set_index("STAID")["geometry"].apply(lambda g: g.y)
sta_lons = stations_gdf.set_index("STAID")["geometry"].apply(lambda g: g.x)

residuals_df["_ndvi_tp"] = residuals_df["date"].apply(quarter_tp)

lookup = {}

for tp in tqdm(
    residuals_df["_ndvi_tp"].unique(), colour="green", position=0, leave=True
):
    path = ndvi_dir / f"ndvi_{tp}.tif"
    if not path.exists():
        print(f"  Missing: {path.name}")
        continue
    print(f"Processing {tp}...")

    with rasterio.open(path) as src:
        nodata = src.nodata
        staids = residuals_df.loc[residuals_df["_ndvi_tp"] == tp, "STAID"].unique()
        lats = sta_lats.reindex(staids).values
        lons = sta_lons.reindex(staids).values

        if src.crs.to_epsg() != 4326:
            xs, ys = transform("EPSG:4326", src.crs, lons, lats)
        else:
            xs, ys = lons, lats

        for i, staid in enumerate(staids):
            row, col = src.index(xs[i], ys[i])
            if 0 <= row < src.height and 0 <= col < src.width:
                val = src.read(1, window=rasterio.windows.Window(col, row, 1, 1))[0, 0]
                lookup[(staid, tp)] = np.nan if val == nodata else val / 254 * 2 - 1
            else:
                lookup[(staid, tp)] = np.nan

residuals_df["ndvi"] = [
    lookup.get((s, t), np.nan)
    for s, t in zip(residuals_df["STAID"], residuals_df["_ndvi_tp"])
]
residuals_df.drop(columns="_ndvi_tp", inplace=True)
print(f"NDVI nulls: {residuals_df['ndvi'].isna().sum()} / {len(residuals_df)}")

  0%|                                                                                                                                                  | 0/23 [00:00<?, ?it/s]

Processing 2019-12-01_2020-03-01...


  4%|██████                                                                                                                                    | 1/23 [00:12<04:27, 12.15s/it]

Processing 2020-03-01_2020-06-01...


  9%|████████████                                                                                                                              | 2/23 [00:23<04:02, 11.56s/it]

...

Processing 2023-06-01_2023-09-01...


 65%|█████████████████████████████████████████████████████████████████████████████████████████▎                                               | 15/23 [02:45<01:27, 10.93s/it]

Processing 2023-09-01_2023-12-01...


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 23/23 [02:56<00:00,  7.69s/it]


  Missing: ndvi_2023-12-01_2024-03-01.tif
  Missing: ndvi_2024-03-01_2024-06-01.tif
  Missing: ndvi_2024-06-01_2024-09-01.tif
  Missing: ndvi_2024-09-01_2024-12-01.tif
  Missing: ndvi_2024-12-01_2025-03-01.tif
  Missing: ndvi_2025-03-01_2025-06-01.tif
  Missing: ndvi_2025-06-01_2025-09-01.tif
NDVI nulls: 3415611 / 10467039


## 6. Add distance to sea

In [9]:
gadm = gpd.read_file(DATA / "gadm_410_europe.gpkg")
land = gadm.dissolve().to_crs("EPSG:3035")
coastline = land.boundary.iloc[0]

stations_3035 = stations_gdf.to_crs("EPSG:3035")
stations_gdf["dist_to_sea_km"] = stations_3035.geometry.distance(coastline) / 1000

residuals_df = residuals_df.merge(
    stations_gdf[["STAID", "dist_to_sea_km"]],
    on="STAID",
    how="left",
)
print(
    f"dist_to_sea_km nulls: {residuals_df['dist_to_sea_km'].isna().sum()} / {len(residuals_df)}"
)
print(stations_gdf[["STAID", "dist_to_sea_km"]].describe())

dist_to_sea_km nulls: 0 / 10467039
              STAID  dist_to_sea_km
count   8568.000000    8.568000e+03
mean   14056.475840    2.016112e+02
std    10129.389736    5.054822e+02
min        1.000000    5.112265e-08
25%     4581.750000    1.350904e+01
50%    11364.500000    6.084180e+01
75%    24815.500000    1.808102e+02
max    28077.000000    4.401065e+03


## 7. Add altitude

In [10]:
residuals_df = residuals_df.merge(
    stations_gdf[["STAID", "HGHT"]].rename(columns={"HGHT": "altitude"}),
    on="STAID",
    how="left",
)
print(f"altitude nulls: {residuals_df['altitude'].isna().sum()} / {len(residuals_df)}")
print(residuals_df["altitude"].describe())

altitude nulls: 0 / 10467039
count    1.046704e+07
mean     3.417090e+02
std      4.266523e+02
min     -9.999000e+03
25%      4.000000e+01
50%      1.760000e+02
75%      5.020000e+02
max      3.437000e+03
Name: altitude, dtype: float64


## 8. Add DEGURBA (degree of urbanisation)

In [11]:
degurba_path = (
    OUT
    / "GHS_SMOD_E2020_GLOBE_R2023A_54009_1000_V2_0/GHS_SMOD_E2020_GLOBE_R2023A_54009_1000_V2_0.tif"
)

with rasterio.open(degurba_path) as src:
    nodata = src.nodata
    lons = stations_gdf.geometry.x.values
    lats = stations_gdf.geometry.y.values

    # Reproject station coords from WGS84 to the raster CRS (Mollweide)
    xs, ys = transform("EPSG:4326", src.crs, lons, lats)

    degurba_vals = []
    for i in range(len(stations_gdf)):
        row, col = src.index(xs[i], ys[i])
        if 0 <= row < src.height and 0 <= col < src.width:
            val = src.read(1, window=rasterio.windows.Window(col, row, 1, 1))[0, 0]
            degurba_vals.append(np.nan if val == nodata else int(val))
        else:
            degurba_vals.append(np.nan)

stations_gdf["degurba"] = degurba_vals

residuals_df = residuals_df.merge(
    stations_gdf[["STAID", "degurba"]],
    on="STAID",
    how="left",
)
print(f"degurba nulls: {residuals_df['degurba'].isna().sum()} / {len(residuals_df)}")
print(stations_gdf["degurba"].value_counts().sort_index())

degurba nulls: 0 / 10467039
degurba
10     248
11    3260
12    2252
13     672
21     832
22     145
23     467
30     692
Name: count, dtype: int64


## 8. Save

In [14]:
DEGURBA_LABELS = {
  10: "Water",
  11: "Very low density rural",
  12: "Low density rural",
  13: "Rural cluster",
  21: "Suburban",
  22: "Semi-dense urban cluster",
  23: "Dense urban cluster",
  30: "Urban centre",
}

residuals_df["degurba_label"] = residuals_df["degurba"].map(DEGURBA_LABELS)

residuals_df.to_parquet(OUT / "residuals.parquet", index=False)
print(f"Saved {len(residuals_df):,} rows to {OUT / 'residuals.parquet'}")
residuals_df.info()
residuals_df.head()

Saved 10,467,039 rows to /home2/lobib/gen_hack/data2/residuals.parquet
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10467039 entries, 0 to 10467038
Data columns (total 12 columns):
 #   Column           Dtype         
---  ------           -----         
 0   STAID            int64         
 1   date             datetime64[ns]
 2   station_celsius  float64       
 3   era5_celsius     float32       
 4   residual         float64       
 5   tp               float32       
 6   wind             float32       
 7   ndvi             float64       
 8   dist_to_sea_km   float64       
 9   altitude         int64         
 10  degurba          int64         
 11  degurba_label    object        
dtypes: datetime64[ns](1), float32(3), float64(4), int64(3), object(1)
memory usage: 838.5+ MB


,STAID,date,station_celsius,era5_celsius,residual,tp,wind,ndvi,dist_to_sea_km,altitude,degurba,degurba_label
0,2,2020-01-01,5.4,3.112396,2.287604,3.236035e-07,2.935515,0.259843,85.605731,160,21,Suburban
1,2,2020-01-02,6.3,4.385101,1.914899,1.831601e-06,3.986341,0.259843,85.605731,160,21,Suburban
2,2,2020-01-03,6.8,6.099335,0.700665,1.481518e-04,4.734711,0.259843,85.605731,160,21,Suburban
3,2,2020-01-04,2.7,1.308136,1.391864,8.525351e-06,4.175328,0.259843,85.605731,160,21,Suburban
4,2,2020-01-05,0.9,1.942871,-1.042871,5.183578e-04,2.081413,0.259843,85.605731,160,21,Suburban
